# Golden HOUR V3 — Investigator View in Jupyter

Use this notebook **inside the same IBM LinuxONE Jupyter website** as your running V3 server. It talks to `http://127.0.0.1:8765` **from the Python kernel**, not directly from your laptop. No SSH, public port, or Jupyter server proxy is needed.

1. Keep the terminal running `python -m v3.app --host 127.0.0.1 --port 8765`.
2. Run the connection test below.
3. Run the interactive panel cell. Choose an account, press **Generate explanation**, and later press **Refresh** to see the result.

**Safety:** Mock fixtures are labelled as mock. Approve/reject buttons record simulated reviews only; they do not affect bank accounts. Don't use real identifying information in this demo.

In [ ]:
import json
from urllib.request import urlopen

BASE_URL = "http://127.0.0.1:8765"
with urlopen(BASE_URL + "/api/health", timeout=5) as response:
    health = json.load(response)
print("V3 server connection:", health)
with urlopen(BASE_URL + "/api/state", timeout=5) as response:
    state = json.load(response)
print("Live state ready:", state.get("ready"), "checkpoint:", state.get("sequence"))

## View the live state without installing anything

Run the next cell to see a browser-friendly snapshot. **Rerun the cell** to refresh it after the mock feed changes. This works even if `ipywidgets` is not installed. The optional interactive panel is further below.

In [ ]:
from IPython.display import display, HTML
from html import escape
from urllib.request import urlopen
import json

with urlopen("http://127.0.0.1:8765/api/state", timeout=5) as response:
    live = json.load(response)
if not live.get("ready"):
    display(HTML("<h3>Waiting for first mock/live checkpoint</h3><p>Run the mock feed, then rerun this cell.</p>"))
else:
    records = live.get("recommendations", [])
    display(HTML(f"<h2>Golden HOUR V3</h2><p><b>Mode:</b> {escape(str(live.get('mode')))} &nbsp; <b>As-of:</b> {escape(str(live['as_of']))} &nbsp; <b>Checkpoint:</b> {live['sequence']}</p>"
                 f"<p><b>Observed transfers:</b> {live.get('total_seen',0)} &nbsp; <b>Watchlisted accounts:</b> {live.get('watch_count',0)} &nbsp; <b>Proposed holds:</b> {len(records)} / 10</p>"))
    for r in records:
        e = r.get("explanation", {})
        display(HTML("<hr><p><b>Guardian proposed review:</b> <code>" + escape(str(r['acct'])) +
                     "</code> &nbsp; <b>Risk:</b> " + f"{r['risk_score']:.3f}" + "</p><p><b>Evidence:</b> " +
                     escape(str(r.get('reason_facts',''))) + "</p><p><b>Explanation status:</b> " +
                     escape(str(e.get('status','not_requested'))) + "</p><p>" +
                     escape(str(e.get('text','No explanation requested yet'))) + "</p>"))

### Request an explanation without the optional widgets

Run the following cell once to request an explanation for the first current recommendation. After 10–20 seconds, rerun the live-state cell above to see the draft or fallback. This runs via the server-side Python kernel, so it does not require a proxy.

In [ ]:
from urllib.request import Request, urlopen
import json

with urlopen("http://127.0.0.1:8765/api/state", timeout=5) as response:
    current = json.load(response)
recs = current.get("recommendations", [])
if recs:
    rec = recs[0]
    body = json.dumps({"acct":rec["acct"], "decision_id":rec["decision_id"]}).encode()
    req = Request("http://127.0.0.1:8765/api/explain", data=body,
                  headers={"Content-Type":"application/json"}, method="POST")
    with urlopen(req,timeout=5) as response:
        print("Explanation request:",json.load(response))
    print("Rerun the live-state cell in 10–20 seconds to see the result.")
else:
    print("No Guardian recommendations yet. Run the mock feed first.")

## Optional interactive investigator panel (requires ipywidgets)

This is a Jupyter-native view, not the standalone `dashboard.html` website. It uses the **same underlying V3 server**, including its simulated approval audit log and on-demand Granite worker. The model is lazy-loaded; your first request might take longer. Press **Refresh** after a few seconds.

In [ ]:
import json
from html import escape
from urllib.request import Request, urlopen
from urllib.error import HTTPError
from IPython.display import display, clear_output
try:
    import ipywidgets as W
except ImportError:
    print("ipywidgets is not installed in this notebook kernel. Please tell me; don't install packages on the shared server yet.")
    raise

BASE_URL = "http://127.0.0.1:8765"

def api(path, payload=None):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    req = Request(BASE_URL + path, data=data,
                  headers={"Content-Type": "application/json"} if data else {},
                  method="POST" if data is not None else "GET")
    try:
        with urlopen(req, timeout=6) as response:
            return json.load(response)
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"HTTP {exc.code}: {detail[:350]}") from exc

S = {"data": {}}
header = W.HTML()
overview = W.HTML()
feed = W.HTML()
account = W.Dropdown(description="Review:", layout=W.Layout(width="100%"))
details = W.HTML()
messages = W.Output()
refresh_button = W.Button(description="Refresh", icon="refresh", button_style="info")
explain_button = W.Button(description="Generate Granite draft", icon="comment")
approve_button = W.Button(description="Approve simulated hold", icon="check", button_style="success")
reject_button = W.Button(description="Reject proposed hold", icon="times", button_style="warning")

def table(headers, rows):
    styles = "border-bottom:1px solid #d0d7de;padding:7px 9px;text-align:left;vertical-align:top"
    th = "".join(f'<th style="{styles}">{escape(str(h))}</th>' for h in headers)
    body = "".join("<tr>" + "".join(f'<td style="{styles}">{escape(str(v))}</td>' for v in row) + "</tr>" for row in rows)
    return f'<table style="border-collapse:collapse;width:100%;font-size:13px"><thead><tr>{th}</tr></thead><tbody>{body}</tbody></table>'

def selected_rec():
    key = account.value
    return next((r for r in S["data"].get("recommendations", []) if r.get("decision_id") == key), None)

def show_details(*_):
    r = selected_rec()
    if not r:
        details.value = '<p>Select a proposed account hold to inspect its evidence.</p>'
        return
    expl = r.get("explanation", {})
    state = escape(str(expl.get("status", "not_requested")))
    draft = escape(str(expl.get("text", "Not yet generated. Evidence is available immediately.")))
    source = escape(str(expl.get("source", "")))
    details.value = (
        '<section style="padding:12px;border:1px solid #d0d7de;border-radius:10px">'
        f'<p><b>Account:</b> <code>{escape(r["acct"])}</code> &nbsp; '
        f'<b>Risk score:</b> {float(r["risk_score"]):.3f} &nbsp; '
        f'<b>Review:</b> {escape(str(r.get("review_status", "awaiting")))}</p>'
        f'<p><b>Decision as of:</b> {escape(str(r["as_of"]))}</p>'
        f'<p><b>Decision-time facts:</b> {escape(str(r.get("reason_facts", "")))}</p>'
        f'<p><b>Explanation status:</b> {state} {("(" + source + ")") if source else ""}</p>'
        f'<p><b>AI draft / fallback:</b> {draft}</p>'
        '<p><i>Review required. No real account hold is applied.</i></p></section>'
    )

def refresh(_=None):
    try:
        current = api("/api/state")
        S["data"] = current
        if not current.get("ready"):
            header.value = "<h2>Golden HOUR V3 — Waiting for mock/live snapshots</h2>"
            overview.value = "<p>Run the mock feed in your second Jupyter terminal.</p>"
            feed.value = ""
            account.options = []
            show_details()
            return
        mode = str(current.get("mode", "unverified"))
        header.value = (f'<h2>Golden HOUR V3 — Investigator Review</h2>'
                        f'<p><b>Mode:</b> {escape(mode)} · <b>Simulated time:</b> '
                        f'{escape(str(current.get("as_of")))} · <b>Checkpoint:</b> {current.get("sequence")}</p>')
        recs = current.get("recommendations", [])
        overview.value = (f'<p><b>Observed transactions:</b> {current.get("total_seen", 0)} &nbsp; '
                          f'<b>Watchlist:</b> {current.get("watch_count", 0)} &nbsp; '
                          f'<b>Proposed holds:</b> {len(recs)} / 10</p>'
                          '<h4>Observed cross-bank links</h4>' + table(
                              ["From", "To", "Transfers"],
                              [(x.get("from_bank"), x.get("to_bank"), x.get("count")) for x in current.get("bank_links", [])]
                          ) + '<h4>Recent transactions</h4>' + table(
                              ["Time", "From", "To", "From bank", "To bank"],
                              [(x.get("timestamp"), x.get("from_acct"), x.get("to_acct"), x.get("from_bank"), x.get("to_bank")) for x in current.get("recent_transactions", [])[-10:]]
                          ))
        feed.value = ('<h4>High-risk watchlist (not proposed holds)</h4>' + table(
            ["Account", "Risk score"],
            [(x.get("acct"), f'{float(x["risk_score"]):.3f}') for x in current.get("watchlist", [])[:12]]
        ))
        chosen = account.value
        opts = [(f'#{r["rank"]} · {r["acct"]} · risk {r["risk_score"]:.3f}', r["decision_id"]) for r in recs]
        account.options = opts
        if chosen in [x[1] for x in opts]:
            account.value = chosen
        show_details()
    except Exception as exc:
        with messages:
            clear_output(wait=True)
            print("Refresh error:", exc)

def action(kind):
    rec = selected_rec()
    if not rec:
        with messages:
            clear_output(wait=True)
            print("There is no current recommendation to review.")
        return
    data = {"acct":rec["acct"], "decision_id":rec["decision_id"]}
    if kind in ("approve", "reject"):
        data["action"] = kind
    try:
        result = api("/api/explain" if kind == "explain" else "/api/decision", data)
        with messages:
            clear_output(wait=True)
            print("Server response:", result)
            if kind == "explain":
                print("The model is running in the server background. Click Refresh after 10–20 seconds.")
        refresh()
    except Exception as exc:
        with messages:
            clear_output(wait=True)
            print("Action failed:", exc)

account.observe(show_details, names="value")
refresh_button.on_click(refresh)
explain_button.on_click(lambda _: action("explain"))
approve_button.on_click(lambda _: action("approve"))
reject_button.on_click(lambda _: action("reject"))

display(W.VBox([
    header, overview, feed,
    W.HTML('<h3>Guardian V3 — Proposed holds (human review required)</h3>'),
    account, details,
    W.HBox([refresh_button, explain_button]),
    W.HBox([approve_button, reject_button]),
    messages,
]))
refresh()